# Bilingual page matching: three real pages

This notebook replays the saved pilot without network requests or manual match choices.

1. Compare with entities independently collected from the English and Japanese articles.
2. Compare unresolved mentions with IDs matched in step 1, across the three pages.
3. Compare remaining mentions with the freshly collected WWII Wikidata population.
4. Give each remaining mention its own synthetic ID.

Use the highest compatible trigram Dice score, at least **0.80**. Equal-highest QID ties use a saved random seed. A score is not a probability of a correct match.

Article collection uses links plus automatic English capitalized phrases, Japanese dictionary noun phrases, and year strings. Unlinked phrases use batched exact English/Japanese Wikidata label and alias lookups. All exact-name candidates are kept; homonymous candidates can include unrelated IDs. Full body, tables and bibliography are traversed. **Extraction is not guaranteed exhaustive.** The Indian division has no Japanese sitelink in the fetched record. The WWII population is the exact saved query, not every entity that could be related to WWII.

In [1]:
from pathlib import Path
from collections import Counter
import json, random, sys
import pandas as pd

HERE = Path.cwd()
if (HERE / "GG_Data/entity_resolution").exists():
    HERE = HERE / "GG_Data/entity_resolution"
sys.path.insert(0, str(HERE))
from bilingual_page_pilot import read, choose, categories, synthetic_id, digest
from generate_candidates import NameIndex, normalize, trigrams, similarity

RUN = HERE / "runs/bilingual_pilot_20261018"
manifest = read(RUN / "run_manifest.json")
assert manifest["status"] == "COMPLETED"
assert digest(HERE / "bilingual_page_pilot.py") == manifest["code_sha256"]
assert all(digest(RUN / name) == sha for name, sha in manifest["outputs"].items())
saved = read(RUN / "matches.json.gz")
metadata = read(RUN / "entity_metadata.json.gz")
world = read(RUN / "fresh_wwii_names.json.gz")
articles = read(RUN / "articles_and_page_inventory.json.gz")
THRESHOLD = 0.80  # Try 0.90 or 1.00. Lower values need additional type records.
SEED = manifest["seed"]
PAGE = "1st Mobile Fleet (Imperial Japanese Navy)"
assert 0.80 <= THRESHOLD <= 1.00
print(f"{len(saved)} mentions; {len(world):,} fresh WWII IDs")

362 mentions; 103,235 fresh WWII IDs


## 1. Two separate lists: Daniel mentions and independent page entities
Change `PAGE` above and run the cells again. Repeated mentions remain separate rows.

In [2]:
daniel = pd.DataFrame([{"mention_id": r["mention_id"], "word": r["raw"]["word"], "type": r["raw"]["entity_group"]} for r in saved if r["raw"]["Source"] == PAGE])
display(daniel)

pools = {page: sorted({q for a in articles if a["daniel_source"] == page for q in a["pool"]}) for page in manifest["pages"]}
known = pd.DataFrame([{"QID": q, "English": metadata[q]["labels"].get("en", ""), "Japanese": metadata[q]["labels"].get("ja", ""), "compatible_types": sorted(categories(metadata[q], metadata))} for q in pools[PAGE]])
print("Independent page pool:", len(known), "QIDs. English and Japanese names use the same QID.")
display(known)


,mention_id,word,type
0,mention-1b9c25878d9fac32de86bd65e9df32782bf1af...,1st Mobile Fleet,ORG
1,mention-e8b98c44e909d8664ab23fc19751a98df68d67...,Dai,ORG
2,mention-cf230a83965fe54419641becdf1647a7aedc36...,Ichi,MISC
3,mention-c81b1fb4004de4fc623eaf59f5912d64ef7a35...,Kidō,ORG
4,mention-341950c17af0849792217706f6fafd91408952...,Imperial Japanese Navy,ORG
5,mention-a684a002d53cb39a985a5579363c38b8ab4d18...,World War II,MISC
6,mention-c96c0c9edb3a5ab6d23fa21092e293eab25d61...,Commander in chief,PER
7,mention-3b097bed8fac618532f454be9c6686399ec771...,Chief of staff,PER
8,mention-a73fed3805993a18af6e9df6997687bbbd9512...,Maru Special,MISC
9,mention-af0b7514f28ed8ad65a2617f5a2dab4815bc46...,Ushio Shobō,MISC


Independent page pool: 375 QIDs. English and Japanese names use the same QID.


,QID,English,Japanese,compatible_types
0,P121,item operated,次を運用する,[MISC]
1,P607,participated in conflict,参加した戦い,[MISC]
2,Q1,universe,宇宙,[MISC]
3,Q100287591,The World,,[MISC]
4,Q100287592,The World,,[MISC]
...,...,...,...,...
370,Q946846,Japanese destroyer Asashimo,朝霜,[MISC]
371,Q951459,Ryūhō,龍鳳,[MISC]
372,Q960430,Fleet,,[LOC]
373,Q96733496,World,,[MISC]


## 2. See the name score
Change the two strings. The score is twice the number of shared trigrams divided by the total number of trigrams. Short names use the existing notebook fallback.

In [3]:
LEFT = "Ushio Shobo"
RIGHT = "Ushio Shobou"
a, b = trigrams(LEFT), trigrams(RIGHT)
print("Normalized:", normalize(LEFT), "|", normalize(RIGHT))
print("Left:", sorted(a))
print("Right:", sorted(b))
print("Shared:", sorted(a & b))
print("Score:", similarity(LEFT, RIGHT))

Normalized: ushio shobo | ushio shobou
Left: [' sh', 'hio', 'hob', 'io ', 'o s', 'obo', 'shi', 'sho', 'ush']
Right: [' sh', 'bou', 'hio', 'hob', 'io ', 'o s', 'obo', 'shi', 'sho', 'ush']
Shared: [' sh', 'hio', 'hob', 'io ', 'o s', 'obo', 'shi', 'sho', 'ush']
Score: 0.9473684210526315


## 3. First pass: page entities
The type check happens before the highest score is chosen. A higher score with a wrong type cannot win.

In [4]:
working = [{"mention_id": r["mention_id"], "raw": r["raw"], "attempts": []} for r in saved]

def step(stage, indexes):
    for row in working:
        if "assigned_id" in row:
            continue
        index = indexes[row["raw"]["Source"]] if isinstance(indexes, dict) else indexes
        candidates = index.find(row["raw"]["word"], "MISC", THRESHOLD)
        result = choose(candidates, row["raw"]["entity_group"], metadata,
                        random.Random(f"{SEED}:{stage}:{row['mention_id']}"))
        row["attempts"].append({"stage": stage, **result})
        if result["status"] == "MATCHED":
            row.update(assigned_id=result["entity_id"], matched_stage=stage,
                       matched_name=result["matched_name"], score=result["name_similarity"])
    return dict(Counter(r.get("matched_stage", "UNRESOLVED") for r in working))

page_indexes = {page: NameIndex(ids, metadata) for page, ids in pools.items()}
step("PAGE_EN_JA", page_indexes)

{'PAGE_EN_JA': 225, 'UNRESOLVED': 137}

## 4. Second pass: only IDs already matched
The pool is frozen after the first pass. The three pages share this pool. No IDs from the old graph are added.

In [5]:
frozen_ids = sorted({r["assigned_id"] for r in working if "assigned_id" in r})
print("Frozen matched pool:", len(frozen_ids), "QIDs")
step("ALREADY_MATCHED", NameIndex(frozen_ids, metadata))

Frozen matched pool: 150 QIDs


{'PAGE_EN_JA': 225, 'UNRESOLVED': 133, 'ALREADY_MATCHED': 4}

## 5. Third pass: fresh WWII reference
This set was collected from current Wikidata statements for WWII, its sub-conflicts, participation, parts and main subject. The exact query and retrieval times are in the run manifest. It can still omit entities that are relevant in meaning but lack these statements.

In [6]:
wwii_index = NameIndex(sorted(world), world)
step("FRESH_WWII", wwii_index)

{'PAGE_EN_JA': 225, 'UNRESOLVED': 122, 'FRESH_WWII': 11, 'ALREADY_MATCHED': 4}

## 6. Separate synthetic ID for each unresolved mention
`SYN_TYPE_` marks incompatible types. `SYN_TYPE_UNKNOWN_` marks missing type evidence. Other unresolved mentions use `SYN_`. Synthetic IDs do not prove that a new real-world entity exists.

In [7]:
for row in working:
    if "assigned_id" not in row:
        reasons = {a["status"] for a in row["attempts"]}
        reason = "TYPE_MISMATCH" if "TYPE_MISMATCH" in reasons else "TYPE_UNKNOWN" if "TYPE_UNKNOWN" in reasons else "NO_NAME_ABOVE_THRESHOLD"
        row.update(assigned_id=synthetic_id(row, reason), matched_stage="SYNTHETIC", unresolved_reason=reason)

result = pd.DataFrame([{**r["raw"], "assigned_id": r["assigned_id"], "stage": r["matched_stage"], "matched_name": r.get("matched_name", ""), "similarity": r.get("score"), "unresolved_reason": r.get("unresolved_reason", "")} for r in working])
display(result.groupby(["Source", "stage"]).size().unstack(fill_value=0))
display(result[result.Source == PAGE][["word", "entity_group", "assigned_id", "stage", "matched_name", "similarity", "unresolved_reason"]])
assert len(working) == len(saved)
if THRESHOLD == .80 and SEED == manifest["seed"]:
    assert [r["assigned_id"] for r in working] == [r["assigned_id"] for r in saved]
    print("PASS: exact replay of all saved assigned IDs")

stage,ALREADY_MATCHED,FRESH_WWII,PAGE_EN_JA,SYNTHETIC
Source,,,,
14th Indian Infantry Division,4,10,111,68
1943 Cairo Declaration,0,1,109,45
1st Mobile Fleet (Imperial Japanese Navy),0,0,5,9


,word,entity_group,assigned_id,stage,matched_name,similarity,unresolved_reason
348,1st Mobile Fleet,ORG,Q4596490,PAGE_EN_JA,1st Mobile Fleet,1.0,
349,Dai,ORG,SYN_e8b98c44e909d8664ab23fc1,SYNTHETIC,,NaN,NO_NAME_ABOVE_THRESHOLD
350,Ichi,MISC,SYN_cf230a83965fe54419641bec,SYNTHETIC,,NaN,NO_NAME_ABOVE_THRESHOLD
351,Kidō,ORG,SYN_c81b1fb4004de4fc623eaf59,SYNTHETIC,,NaN,NO_NAME_ABOVE_THRESHOLD
352,Imperial Japanese Navy,ORG,Q219712,PAGE_EN_JA,Imperial Japanese Navy,1.0,
353,World War II,MISC,Q362,PAGE_EN_JA,World War II,1.0,
354,Commander in chief,PER,SYN_TYPE_c96c0c9edb3a5ab6d23fa210,SYNTHETIC,,NaN,TYPE_MISMATCH
355,Chief of staff,PER,SYN_3b097bed8fac618532f454be,SYNTHETIC,,NaN,NO_NAME_ABOVE_THRESHOLD
356,Maru Special,MISC,SYN_a73fed3805993a18af6e9df6,SYNTHETIC,,NaN,NO_NAME_ABOVE_THRESHOLD
357,Ushio Shobō,MISC,SYN_af0b7514f28ed8ad65a2617f,SYNTHETIC,,NaN,NO_NAME_ABOVE_THRESHOLD


PASS: exact replay of all saved assigned IDs


## 7. Inspect every decision for one mention
Change `MENTION`. Each rejected candidate keeps its name, score and ID; its reference type is shown here.

In [8]:
MENTION = "Commander-in-chief"
for row in working:
    if row["raw"]["Source"] == PAGE and row["raw"]["word"] == MENTION:
        print(json.dumps(row, ensure_ascii=False, indent=2))
        for attempt in row["attempts"]:
            for candidate in attempt.get("rejected", []):
                qid = candidate["entity_id"]
                print(qid, metadata[qid]["labels"], sorted(categories(metadata[qid], metadata)))
